In [1]:
print(123)

123


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url = "https://api.groq.com/openai/v1"
)

In [3]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = [{"role":"user", "content": prompt}]
    )
    return response.choices[0].message.content

In [4]:
llm("Hey, what's up")

"Not much, just here and ready to chat. How about you? How's your day going so far?"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm excited you're interested in the course. However, I need a bit more information. Could you please tell me which course you're referring to? Additionally, what's the current status of the course (is it ongoing, about to start, or has it already ended)? This will help me provide a more accurate answer to your question.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [10]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [11]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)    

1350

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
# score = sim(query, document)

In [14]:
from minsearch import Index

index = Index(
    text_fields = ['question', "sections", "answer"],
    keyword_fields = ["course"]
)
index.fit(documents)

In [15]:
question =  "I just discovered the course. Can I join now?"
search_results = index.search(
    question,
    boost_dict = {"question": 2.0 ,"section": 0.5},
    filter_dict = {"course": "llm-zoomcamp"},
    num_results = 5,
     )
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '1e829a8c6f',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'Do I need a new GitHub repo for Module 2, or just a new codespace?',
  'answer': "Just a new codespace

In [16]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Can I run the course locally instead of Codespaces?',
 'Do I need a new GitHub repo for Module 2, or just a new codespace?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [17]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [18]:
results = index.search(
     question,
     num_results = 5,
     filter_dict = {"course": "mlops-zoomcamp"}
     )
[doc['question'] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [ ]:
# def search(question, course="llm-zoomcamp"):
#     boost_dict = {"question": 2.0, "section": 0.5}
#     filter_dict = {"course": course}

#     return index.search(
#        # question,
#         boost_dict=boost_dict,
#         filter_dict=filter_dict,
#         num_results=5
#     )

In [20]:
search_results = search(question)

In [21]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [25]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Module 1: RAG
Q: Can I run the course locally instead of Codespaces?
A: Yes. Codespaces is just the easiest way for everyone to start with the same environment.

You can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.

If you run locally, make sure you document your setup and keep your environment reproducible.

Module 2: Vector Search
Q: Do I need a new GitHub repo for Module 2, or just a new codespace?
A: Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you create it from your existing repository, so you don't need a new GitHub repo.

Use a separate codespace for Module 

In [26]:
chat.completions.output[0]

NameError: name 'chat' is not defined

In [ ]:
response =  openai_client.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = [{"role":"user", "content": prompt}]
)



# def llm(prompt):
# response = openai_client.chat.completions.create(
#         model = "llama-3.3-70b-versatile",
#         messages = [{"role":"user", "content": prompt}]
#     )
#     return response.choices[0].message.content

In [ ]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))]

In [ ]:
response.choices[0].message.content

'Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [ ]:
response.choices[0].message.content

'Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [ ]:
response.usage


CompletionUsage(completion_tokens=34, prompt_tokens=533, total_tokens=567, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.054806274, prompt_time=0.027357206, completion_time=0.05849393, total_time=0.085851136)

In [ ]:
# def llm(instructions, user_prompt, model="llama-3.3-70b-versatile"):
#     message_history = [
#         {"role": "system", "content": instructions},
#         {"role": "user", "content": user_prompt}
#     ]

#     response = openai_client.chat.completions.create(
#         model=model,
#         messages=message_history
#     )

#     return response.choices[0].message.content

In [ ]:
# def rag(query, model="llama-3.3-70b-versatile"):
#     search_results = search(query)
#     prompt = build_prompt(query, search_results)
#     answer = llm(INSTRUCTIONS, prompt, model=model)
#     return answer

In [ ]:
# answer = rag("I just discovered the course. Can I join now?")
# print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while project submissions are still being accepted.


In [ ]:
# rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a "live" cohort and pass the Capstone project. Additionally, you need to peer-review 3 capstone projects after submitting your own project. Homework is not mandatory, but it is recommended for reinforcing concepts. \n\nPlease note that certificates are not awarded for the self-paced mode. Also, make sure to update your official name in the "Edit Course Profile" section if you want it to appear on your certificate instead of your default nickname.'

In [7]:
from dotenv import load_dotenv
load_dotenv

from ingest import load_faq_data , build_index
from rag_helper import RAGBase
from openai import OpenAI
import os
documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI(
    api_key = os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(index = index, llm_client = openai_client)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join the course. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [8]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [9]:
print(assistant.rag("How do I get a certificate?"))
print(assistant.rag("Can I still join the course after it started?"))

To get a certificate, you must finish the course with a "live" cohort and pass the Capstone project. Additionally, you will need to peer-review 3 capstone projects after submitting your own project. Homework is not mandatory, but it is recommended for reinforcing concepts.
Yes, you can still join the course after it started, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
